In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip freeze > '/content/drive/MyDrive/Colab Notebooks/pos-ifg/trabalho_modulo_2/8.regressao_linear/arquivos_gerados/rl_energia_solar_requirements.txt'

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import numpy as np
import folium

In [ ]:
# --- Etapa 1: Carrega e Prepara os Dados de Treinamento ---

print("--- Iniciando Etapa 1: Preparação dos Dados para Regressão ---")
df_train = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/pos-ifg/trabalho_modulo_2/8.regressao_linear/FACT_LOCALIDADE_CLIMA_202509272358.csv')

# Passando os nomes das colunas para minúsculas (lowercase)
df_train.columns = [col.replace('"', '').lower() for col in df_train.columns]

# Remove dados faltantes (NaN)
print("\nLimpando dados faltantes no arquivo de treino...")
df_train.dropna(inplace=True)
print("Limpeza do arquivo de treino concluída.")


--- Iniciando Etapa 1: Preparação dos Dados para Regressão ---

Limpando dados faltantes no arquivo de treino...
Limpeza do arquivo de treino concluída.


In [ ]:
# --- Etapa 2: Treinamento do Modelo de Regressão ---

print("\n--- Iniciando Etapa 2: Treinamento do Modelo de Regressão ---")

# Aqui foi definida a variável alvo e as variáveis de entrada (features)
target = 'shortwave_radiation_sum'
features = [
    'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean',
    'relative_humidity_2m_mean', 'surface_pressure_mean', 'wind_speed_10m_max',
    'wind_speed_10m_mean', 'precipitation_sum'
]

# Features que serão usadas
print("\nFeatures que serão usadas no modelo:")
print(features)

X = df_train[features]
y = df_train[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalização dos dados
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("\nDados de treino e teste normalizados.")

print("\nIniciando o treinamento do modelo...")
model = LinearRegression(n_jobs=-1)
model.fit(X_train_scaled, y_train)
print("Treinamento concluído.")

# Avaliação da performance do modelo
y_pred = model.predict(X_test_scaled)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"\nResultados da Avaliação do Modelo de Regressão Linear:")
print(f"R² (Coeficiente de Determinação): {r2:.4f}")
print(f"RMSE (Raiz do Erro Quadrático Médio): {rmse:.4f} MJ/m²")



--- Iniciando Etapa 2: Treinamento do Modelo de Regressão ---

Features que serão usadas no modelo:
['temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean', 'relative_humidity_2m_mean', 'surface_pressure_mean', 'wind_speed_10m_max', 'wind_speed_10m_mean', 'precipitation_sum']

Dados de treino e teste normalizados.

Iniciando o treinamento do modelo...
Treinamento concluído.

Resultados da Avaliação do Modelo de Regressão Linear:
R² (Coeficiente de Determinação): 0.4654
RMSE (Raiz do Erro Quadrático Médio): 2.9952 MJ/m²


In [ ]:
# --- Etapa 3: Salva o Modelo Treinado ---

print("\n--- Iniciando Etapa 3: Salvando o Modelo de Regressão e o Scaler ---")
model_filename = '/content/drive/MyDrive/Colab Notebooks/pos-ifg/trabalho_modulo_2/8.regressao_linear/arquivos_gerados/linear_regression_solar.joblib'
scaler_filename = '/content/drive/MyDrive/Colab Notebooks/pos-ifg/trabalho_modulo_2/8.regressao_linear/arquivos_gerados/scaler_solar.joblib'
joblib.dump(model, model_filename)
joblib.dump(scaler, scaler_filename)
print(f"Modelo salvo como: {model_filename}")
print(f"Scaler salvo como: {scaler_filename}")



--- Iniciando Etapa 3: Salvando o Modelo de Regressão e o Scaler ---
Modelo salvo como: /content/drive/MyDrive/Colab Notebooks/pos-ifg/trabalho_modulo_2/8.regressao_linear/arquivos_gerados/linear_regression_solar.joblib
Scaler salvo como: /content/drive/MyDrive/Colab Notebooks/pos-ifg/trabalho_modulo_2/8.regressao_linear/arquivos_gerados/scaler_solar.joblib


In [ ]:
# --- Etapa 4: Usando o Modelo para Prever o Potencial Solar ---

print("\n--- Iniciando Etapa 4: Prevendo a Radiação Solar para Goiás ---")
df_goias = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/pos-ifg/trabalho_modulo_2/8.regressao_linear/goias_clima_era5_diario_completo.csv')

# Limpeza de dados faltantes no arquivo de Goiás, se houvercaso existam
if df_goias[features].isnull().sum().any():
    print("\nAlerta: Encontrados valores NaN nos dados de Goiás. Removendo as linhas correspondentes.")
    df_goias.dropna(subset=features, inplace=True)

# Carregando o scaler que foi treinado com os dados originaid
loaded_scaler = joblib.load(scaler_filename)
X_goias_scaled = loaded_scaler.transform(df_goias[features])

# Carregando o modelo treinado
loaded_model = joblib.load(model_filename)
df_goias['RADIACAO_SOLAR_PREDITA_LR'] = loaded_model.predict(X_goias_scaled)
results_filename_lr = '/content/drive/MyDrive/Colab Notebooks/pos-ifg/trabalho_modulo_2/8.regressao_linear/arquivos_gerados/previsoes_solar_goias_lr.csv'
df_goias.to_csv(results_filename_lr, index=False)
print(f"Previsões salvas em: {results_filename_lr}")



--- Iniciando Etapa 4: Prevendo a Radiação Solar para Goiás ---

Alerta: Encontrados valores NaN nos dados de Goiás. Removendo as linhas correspondentes.
Previsões salvas em: /content/drive/MyDrive/Colab Notebooks/pos-ifg/trabalho_modulo_2/8.regressao_linear/arquivos_gerados/previsoes_solar_goias_lr.csv


In [ ]:
# --- Etapa 5: Carregando arquivo com as previsões ---

print("--- Carregando o arquivo com as previsões ---")
try:
    df_previsoes = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/pos-ifg/trabalho_modulo_2/8.regressao_linear/arquivos_gerados/previsoes_solar_goias_lr.csv')
    print("Arquivo 'previsoes_radiacao_solar_goias.csv' carregado com sucesso.")
except FileNotFoundError:
    print("Erro: O arquivo 'previsoes_radiacao_solar_goias.csv' não foi encontrado.")
    print("Por favor, execute o script anterior para gerar este arquivo ou verifique o caminho.")
    exit()

--- Carregando o arquivo com as previsões ---
Arquivo 'previsoes_radiacao_solar_goias.csv' carregado com sucesso.


In [ ]:
# --- Etapa 6: Calculando a Média de Radiação por Município ---

print("\n--- Calculando o potencial solar médio por município ---")
# Aqui os dados são agrupados por município. Para cada um, é calculada a média da radiação solar.
# Também pegamos a primeira latitude e longitude, já que elas são constantes para cada município.
potencial_solar = df_previsoes.groupby('municipio').agg(
    radiacao_media=('RADIACAO_SOLAR_PREDITA_LR', 'mean'),
    lat=('lat', 'first'),
    lon=('lon', 'first')
).reset_index()


--- Calculando o potencial solar médio por município ---


In [ ]:
# --- Etapa 7: Seleciona os Municípios com Maior Potencial ---

print("--- Selecionando os municípios com maior irradiação ---")
# Ordenamos o resultado pela 'radiacao_media' em ordem decrescente e pegamos os 10 primeiros.
top_municipios = potencial_solar.sort_values(by='radiacao_media', ascending=False).head(10)

print("\nOs 10 municípios com maior potencial solar médio (MJ/m²):")
print(top_municipios.head(10))


--- Selecionando os municípios com maior irradiação ---

Os 10 municípios com maior potencial solar médio (MJ/m²):
                 municipio  radiacao_media        lat        lon
217    São João da Paraúna       21.884377 -16.817832 -50.372148
125                Jaupaci       21.704892 -16.189621 -51.057677
142  Monte Alegre de Goiás       21.643644 -13.294168 -46.923046
110            Israelândia       21.615396 -16.357174 -50.879818
29     Bom Jardim de Goiás       21.547456 -16.174343 -52.076271
157            Nova Crixás       21.547452 -14.133525 -50.561307
22              Arenópolis       21.547198 -16.353199 -51.584059
164          Novo Planalto       21.535202 -13.250585 -49.727995
234                 Uruana       21.487459 -15.577689 -49.642974
24              Aurilândia       21.484572 -16.694744 -50.525310


In [ ]:
# --- Etapa 8: Gera o Mapa Interativo com Folium ---

print("\n--- Gerando o mapa interativo ---")
# Cria um mapa centrado nas coordenadas aproximadas de Goiás. Onde latitude e longitude de Goiás: -15.98, -49.86
mapa_goias = folium.Map(location=[-15.98, -49.86], zoom_start=7, tiles='OpenStreetMap')

# Adiciona um marcador para cada um dos municípios mais relevantes
for index, row in top_municipios.iterrows():
    # Coordenadas do município
    location = [row['lat'], row['lon']]

    # Texto para o pop-up do marcador
    popup_text = f"""
    <b>Município:</b> {row['municipio']}<br>
    <b>Radiação Média:</b> {row['radiacao_media']:.2f} MJ/m²
    """

    # Cria o marcador e o adiciona ao mapa
    folium.Marker(
        text=row['municipio'],
        permanent=True,
        location=location,
        popup=folium.Popup(popup_text, max_width=300),
        tooltip=row['municipio']
    ).add_to(mapa_goias)

# Salva o mapa como um arquivo HTML
mapa_filename = '/content/drive/MyDrive/Colab Notebooks/pos-ifg/trabalho_modulo_2/8.regressao_linear/arquivos_gerados/mapa_potencial_solar_goias.html'
mapa_goias.save(mapa_filename)
print(f"\nMapa salvo com sucesso em '{mapa_filename}'. O mapa também será exibido abaixo.")

mapa_goias


--- Gerando o mapa interativo ---

Mapa salvo com sucesso em '/content/drive/MyDrive/Colab Notebooks/pos-ifg/trabalho_modulo_2/8.regressao_linear/arquivos_gerados/mapa_potencial_solar_goias.html'. O mapa também será exibido abaixo.
